# Sana — a LoRA on Colab (thin shell over the repo runner)

Trains a LoRA for **[Sana](https://github.com/NVlabs/Sana)** (NVlabs; Xie et al. 2024, arXiv 2410.10629) through
model type `sana` in the [AbstractEyes diffusion-pipe fork](https://github.com/AbstractEyes/diffusion-pipe), then
loads the saved LoRA back through diffusers and measures what it does.

**The default run needs no dataset.** The stock Sana 600M 512px model renders its own training set: 192 *upbeat*
images ("a cheerful, upbeat photo of …") captioned with the *neutral* prompt ("a photo of …"). The LoRA can only
lower its loss by making neutral prompts look upbeat, so a working pipeline shows up as a mood shift on prompts
that never mention a mood. A quarter of the subjects never enter training; `s.evaluate()` scores the LoRA on them
with a CLIP mood judge (similarity to upbeat phrases minus similarity to downbeat phrases) and prints a verdict:

| verdict | rule (paired: the LoRA at scale 1 minus no LoRA; 8 held-out subjects × 4 seeds) |
|---|---|
| FLAVOR LORA | mean above 0, at least 75% of the pairs positive, mean above 3 standard errors |
| NO EFFECT | mean within 2 standard errors of 0, or fewer than 60% of the pairs positive |
| MIXED | anything else |

For scale: on the stock model, writing upbeat words into the prompt moves the same score +1.81 over the neutral prompt.

**Same thin-shell contract as the Anima notebooks:** all logic is in the repo
(`geolip_anima_trainer.sana_runner.SanaRunner`); this notebook is clone → install → a few `s.<step>()` calls.
**To change anything, edit the repo and `git pull`; the notebook stays as it is.**

**Before you start:** pick a GPU runtime (the RTX PRO 6000 is the target; the 600M recipe also fits much smaller
cards). An `HF_TOKEN` in Colab Secrets is optional: the models are public, and the token is only used by `s.backup()`.

> **Licences.** Sana's weights are Apache-2.0; its Gemma-2-2B-IT text encoder is under Google's
> [Gemma Terms of Use](https://ai.google.dev/gemma/terms).

## 0 · Confirm the GPU + disk

The default run needs about 25 GB of disk (the ~8.5 GB checkpoint, the dependencies, the images and the LoRA).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
!df -h /content | tail -1

## 1 · Bootstrap — clone + install (→ restart ONCE the first time)

Clones this repo and the diffusion-pipe fork (at `external/diffusion-pipe-fork`) and installs the dependencies (the
logic is in `anima_colab.install`, so a `git pull` updates it with no notebook edit). **Run the cell; if the kernel
restarts, run it again** (it then skips), then continue.

In [ ]:
import os, sys, subprocess
REPO = os.environ.get("ANIMA_REPO", "/content/anima-trainer")
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "https://github.com/AbstractEyes/anima-trainer.git", REPO], check=True)
else:
    subprocess.run(["git", "-C", REPO, "pull", "--ff-only"], check=False)   # the latest runner before importing it
sys.path.insert(0, REPO)
import anima_colab
if anima_colab.install(REPO, dp_url=anima_colab.DP_FORK_URL, similarity=False):   # + the fork that has Sana
    os.kill(os.getpid(), 9)             # -> restart; reconnect and RE-RUN this cell (then it skips)

## 2 · (after restart) Run

Each cell is one `SanaRunner` step. Tunables are keyword arguments on the `SanaRunner(...)` line, for example
`epochs=`, `rank=`, `lr=`, `variant="1600m-1024"`, or `source="folder", dataset_dir=...` for your own images.

In [ ]:
from geolip_anima_trainer.sana_runner import SanaRunner
s = SanaRunner()       # Sana 600M 512px, the built-in upbeat set, rank 32, plain Adam at 1e-4, 10 epochs
s.setup()              # env + optional HF login + GPU check + the fork + the model download (~8.5 GB)

In [ ]:
s.prepare_dataset()    # renders the 192 training images (24 subjects x 8 seeds); 8 subjects stay out of training

In [ ]:
s.build_configs()      # the training tomls; previews of 4 held-out neutral prompts at every save

In [ ]:
s.train()              # blocking; the trainer's log streams here (480 steps). Or s.train(detached=True) + s.tail()

In [ ]:
s.evaluate()           # held-out subjects x 4 seeds: no LoRA vs the LoRA at 0.5 and 1, loaded via diffusers -> verdict + sheet

In [ ]:
s.backup()             # the LoRA + the evaluation -> a private HF model repo (needs HF_TOKEN)

## Notes

- **One-shot:** `SanaRunner().run_all()` does setup → data → configs → train → evaluate → backup.
- **Your own data:** `SanaRunner(source="folder", dataset_dir="/content/my_set", preview_prompts=["..."])`, a folder
  of images with `.txt` captions in natural language (Sana reads sentences). `s.evaluate(["a prompt", ...])` then
  compares no LoRA with the LoRA on your prompts.
- **Other checkpoints:** `variant="600m-1024"`, `"1600m-512"`, `"1600m-1024"` or `"1600m-1024-bf16"`; the training
  resolution follows the checkpoint.
- **Where things land** (`/content/sana_data/`): `datasets/mood_upbeat/` (the training images + captions),
  `configs/`, `runs/sana_lora/<timestamp>/epochN/adapter_model.safetensors` (the LoRA; the previews are in
  `runs/sana_lora/<timestamp>/samples/`), `eval/eval.json` + `eval/sheet.jpg` (rows: held-out subjects; columns: no
  LoRA, scale 0.5, scale 1).
- **Using the LoRA:** on a diffusers `SanaPipeline`,
  `pipe.load_lora_weights("<run>/epochN", weight_name="adapter_model.safetensors")`.
- **Reset-safe:** a fresh runtime just re-runs the cells; the rendered training images are reused.
- **Updating logic = `git pull`** (the bootstrap cell pulls every run); nothing here needs editing.